## Hello World Notebook for C++

This notebook is just a test for the use of [My Binder](https://mybinder.org)

In [ ]:
//%include
#include <functional>
#include <iostream>
#include <utility>
#include <vector>
#include <memory>

A simple example of RAII: perfect forwarding

In [ ]:
class FooClass {
    private:
        std::vector<int> v;
        int n;
    public:
        FooClass (int size) : n(size), v(size,0) {}
        FooClass (int size, std::vector<int>&& vect) {
            n = size;
            v = std::move(vect);
        }
        FooClass (std::vector<int>&& vect) {
            v = std::move(vect);
            n = v.size();
        }
        FooClass (FooClass&& other) noexcept : n(other.n),
        v(std::move(other.v)) {
            other.n = 0;
        }
        FooClass (FooClass& other) = delete;
        FooClass (const FooClass& other) = delete;
        FooClass& operator =(FooClass& other) = delete;
        FooClass& operator =(FooClass&& other) noexcept {
            if (this == &other) return *this;
            this->n = other.n;
            this->v = std::move(other.v);
            other.n = 0;
            return *this;
        }
        void add(FooClass& other) {
            this->n += other.n;
            for (auto x : other.v) v.push_back(x); 
        }
        void print() {
            std::cout << "Size: " << n << " Values: ";
            for (auto x : v) std::cout << x << " ";
            std::cout << std::endl;
        }
};

In [ ]:
//A signature function void(int)
void foo(int x) {std::cout << "foo(" << x <<")\n";}
//A signature function int(void)   
int foo2() {std::cout << "foo2()=" << 0 << "\n"; return 0;}
//A signature function int(int, int)
void foo3(int& x, int& y) {x += y;}
//A signature function void(FooClass,FooClass)
void foo4(FooClass& x1, FooClass& x2) {
    x1.add(x2);
}

This is the variadic template mode, which allows you to capture a list of
arguments and wrap them as a lambda with a signature of `void()`. The template
accepts a type `F` (a `std::function` with a variable signature) and a variable
number of arguments.

In [ ]:
template <typename F, typename... Args>
std::function<void()> create_void_functor(F&& f, Args&&... args)
{
    auto functor = [f=std::forward<F>(f), ...args=std::forward<Args>(args)]()
        mutable -> void{f(args...);};
    return functor;
}

In [ ]:
int main() 
{
    std::function<void(int)> f = foo;
    std::function<int()> f2 = foo2;

    auto f3 = create_void_functor(&foo,5);
    f3();
    auto f4 = create_void_functor(&foo2);
    f4();

    int x = 4;
    int y = 2;

    auto f5 = create_void_functor(&foo3,std::ref(x),std::move(y));
    f5();
    std::cout << "Result: " << x << "\n";

    auto o1 = std::shared_ptr<FooClass>(new FooClass({1,2,3}));
    o1->print();
    auto o2 = std::shared_ptr<FooClass>(new FooClass(1,{10}));
    o2->print();
    auto f6 = create_void_functor(
        [](std::shared_ptr<FooClass> ref1, std::shared_ptr<FooClass> ref2)
            {ref1->add(*ref2);},
            o1,
            std::shared_ptr<FooClass>(new FooClass({4,5,6})));
    f6();
    o1->print();
    return 0;
}